# Sparse Auto-Encoder Trained on ESM-2 Embeddings of Swiss-Prot Proteins

In order to extract generalized features from ESM-2, we need a broad dataset of embeddings to train the Sparse Auto-Encoder (SAE). To do this, we are using the canonical sequences from Swiss-Prot (downloaded 11/21/2024, 2024_05 release). The heme embeddings that we have been playing with up to this point will serve as the test data for later analysis and validation. 

**Imports and Helper Function Definitions**

In [ ]:
# imports
import tensorflow as tf
import keras
from keras import layers
import numpy as np
import pandas as pd
import Bio
from Bio import SeqIO
import transformers
from transformers import AutoTokenizer, TFAutoModel 
from objects.autoencoder import SparseAutoEncoder
import os
from tqdm import tqdm
from csv import writer

# data vis packages
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

### HELPER FUNCTIONS ###

def vector_angle(v1, v2):
    """finds the angle between two vectors"""
    return np.arccos(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def radians_to_degrees(radians):
    return radians * 180 / np.pi

# reduce jagged tensors to smooth out vector arrays
def fix_jagged_2d(embed_list):
    sir_gallahad_the_chaste = []
    # find max length in 2nd dim (sequence length)
    max_len = max([len(arr[0]) for arr in embed_list])
    # append zero vector to embeddings
    for i, arr in enumerate(embed_list):
        if len(arr[0]) < max_len:
            padded = np.append(arr[0], np.zeros((max_len - len(arr[0]), 1280)), axis=0)
            sir_gallahad_the_chaste.append(np.expand_dims(padded, axis=0))
        else:
            sir_gallahad_the_chaste.append(arr)
    return sir_gallahad_the_chaste

# pad sequence strings to a fixed length
def pad_sequences(amino_acid_seq:str, pad_length:int):
    if len(amino_acid_seq)==pad_length:
        return amino_acid_seq
    else:
        return amino_acid_seq+''.join(['-' for i in range(pad_length-len(amino_acid_seq))])

In [ ]:
# load swissprot data and generate embeddings

# file locations and ids
embed_directory = 'data/swissprot/embeddings/'
meta_file = 'data/swissprot/sequence_metadata.csv'
hug_face_id = "facebook/esm2_t33_650M_UR50D"

# make csv file for metadata
if os.path.exists(meta_file):
    raise Exception('metadata file already exists')
else:
    os.system(f'touch {meta_file}')

os.makedirs(embed_directory, exist_ok=True)

# load model
hug_face_id = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(hug_face_id)
ESM_1280 = TFAutoModel.from_pretrained(hug_face_id)

with open('data/swissprot/uniprot_sprot.fasta') as f:
    for entry in tqdm(SeqIO.parse(f, 'fasta'), total=572214):
        
        # get metadata
        sequence = str(entry.seq)
        id = str(entry.id)
        description = (entry.description)

        # generate embeddings
        input = tokenizer(sequence, return_tensors='tf')
        output = ESM_1280(input)
        # save embeddings as numpy binaries
        with open(f"{embed_directory}{id}.npy", "wb") as f:
            np.save(f, np.array(output.last_hidden_state))

        # append metadata to csv
        with open(meta_file, 'a') as f:
            csv_writer = writer(f)
            csv_writer.writerow([id, description, sequence])